# LiH FIG. 13: depolarizing noise + OGM shots + REM

Compiled 6-qubit ansatz with **FIG. 10** initial prep (`R_y(\pi/5)`, CNOT chain, X) then FIG. 13; **LiH** Hamiltonian at bond **2.2**, simple CZ/1Q depolarizing noise, density-matrix `Tr[H ρ]`, and finite-shot energy via **OGM**. Readout is modeled with `p_0_success` / `p_1_success`; **REM** uses the inverse readout matrix via `rem_z_vectors` in `shot_measurement_test_LiH`. No ZNE/CDR.

In [6]:
from __future__ import annotations

import sys
from pathlib import Path


def _test_lih_case_dir() -> Path:
    cwd = Path.cwd().resolve()
    if (cwd / "main_cursor_lib_test_LiH.py").is_file() and (cwd / "shot_measurement_test_LiH.py").is_file():
        return cwd
    sub = cwd / "test_LiH_case"
    if (sub / "main_cursor_lib_test_LiH.py").is_file():
        return sub.resolve()
    for parent in [cwd] + list(cwd.parents):
        cand = parent / "test_LiH_case"
        if (cand / "main_cursor_lib_test_LiH.py").is_file():
            return cand.resolve()
    raise FileNotFoundError("Could not locate test_LiH_case containing main_cursor_lib_test_LiH.py")


def _workspace_with_pauli_ham() -> Path:
    cwd = Path.cwd().resolve()
    for p in [cwd] + list(cwd.parents):
        if (p / "Pauli_Ham").is_dir():
            return p
    return cwd


_CASE = _test_lih_case_dir()
if str(_CASE) not in sys.path:
    sys.path.insert(0, str(_CASE))

import cirq
import numpy as np
import sympy

from main_cursor_lib_test_LiH import (
    GateArityDepolarizingNoise,
    load_hamiltonian_paths,
    load_observable_h,
    prepare_li_h_fig13_ansatz_cirq,
    trace_energy,
)
from shot_measurement_test_LiH import estimate_energy_from_noisy_rho_shots

# LiH at bond 2.2 (six-qubit Pauli file); not the H4 pipeline from repo root.
bond_length = 2.2
H_atom = 2  # placeholder for load_* API when hamiltonian_basename is set
hamiltonian_basename = "LiH"

theta = sympy.Symbol("theta")
theta_value = np.pi / 5

num_shots = 8192
measurement_scheme = "ogm"
sampling_seed = 1234
epsilon = 0.1

p_0_success = np.array([0.9756, 0.9748, 0.9738, 0.9656, 0.9585, 0.9514])
p_1_success = np.array([0.9756, 0.9748, 0.9738, 0.9656, 0.9585, 0.9514])

# Readout: simulate mis-calibration; REM applies inverse (mitigated) channel in post-processing.
apply_readout_noise = True
apply_rem = True  # if True, `energy_rem` uses REM; set False to only report unmitigated shot energy

SHADOWGROUPING_ROOT = "/Users/zacharyhe/shadowgrouping"
ogm_file = Path(f"{SHADOWGROUPING_ROOT}/haozhaowu/LiH/hamil_class/ogm_outputs/OGM_ogm_LiH_{bond_length:.1f}.txt")

random_seed = 1234
depol_prob = DEFAULT_DEPOL_PROB

workspace = _workspace_with_pauli_ham()
print(f"Workspace: {workspace}")
print(f"Bond length: {bond_length} | measurement_scheme={measurement_scheme} | shots={num_shots}")
print(f"OGM file (LiH path, not H4): {ogm_file}")
print(f"OGM file exists: {ogm_file.is_file()}")

Workspace: /Users/zacharyhe/cross_chips_sim
Bond length: 2.2 | measurement_scheme=ogm | shots=8192
OGM file (LiH path, not H4): /Users/zacharyhe/shadowgrouping/haozhaowu/LiH/hamil_class/ogm_outputs/OGM_ogm_LiH_2.2.txt
OGM file exists: True


In [7]:
ansatz_circuit, ansatz_qubits = prepare_li_h_fig13_ansatz_cirq(theta)
observable_H = load_observable_h(
    workspace,
    ansatz_qubits,
    H_atom,
    bond_length,
    hamiltonian_basename=hamiltonian_basename,
)
_, hamiltonian_pkl_path, hamiltonian_text_path = load_hamiltonian_paths(
    workspace, H_atom, bond_length, hamiltonian_basename=hamiltonian_basename
)
print(f"Hamiltonian candidates: {hamiltonian_pkl_path.name}, {hamiltonian_text_path.name}")

resolver = cirq.ParamResolver({theta: float(theta_value)})
hamiltonian_matrix = observable_H.matrix(qubits=ansatz_qubits)
print(f"Parameter: theta = {float(theta_value):.6f}")

Hamiltonian candidates: LiH_bond_2.2.pkl, LiH_bond_2.2_pauli_convention.txt
Parameter: theta = 0.628319


In [3]:
gate_noise = GateArityDepolarizingNoise()
noisy_ansatz = ansatz_circuit.with_noise(gate_noise)
resolved_noisy = cirq.resolve_parameters(noisy_ansatz, resolver)

noisy_simulator = cirq.DensityMatrixSimulator(seed=random_seed)
noisy_result = noisy_simulator.simulate(resolved_noisy, qubit_order=ansatz_qubits)
rho_noisy = noisy_result.final_density_matrix

trace_noisy_energy = trace_energy(hamiltonian_matrix, rho_noisy)
print(f"Tr[H rho_noisy] (gate depolarizing noise): {trace_noisy_energy:.12f} Ha")

Tr[H rho_noisy] (gate depolarizing noise): -0.129169484483 Ha


In [4]:
if not ogm_file.is_file():
    print(
        "Skip OGM shot estimate: OGM file missing. Generate LiH OGM at this bond or set SHADOWGROUPING_ROOT."
    )
elif not Path(SHADOWGROUPING_ROOT).is_dir():
    print("Skip OGM shot estimate: SHADOWGROUPING_ROOT does not exist.")
else:
    shot_est = estimate_energy_from_noisy_rho_shots(
        rho_noisy,
        observable_H,
        ansatz_qubits,
        num_shots=num_shots,
        measurement_scheme=measurement_scheme,
        p_0_success=p_0_success,
        p_1_success=p_1_success,
        apply_rem=apply_rem,
        apply_readout_noise=apply_readout_noise,
        sampling_seed=sampling_seed,
        epsilon=epsilon,
        ogm_file=ogm_file,
        shadowgrouping_root=SHADOWGROUPING_ROOT,
    )
    eu = float(shot_est["energy_unmitigated"])
    er = float(shot_est["energy_rem"])
    print(f"Finite-shot energy (readout noise, no REM correction): {eu:.12f} Ha")
    print(f"Finite-shot energy (REM readout mitigation):          {er:.12f} Ha")
    print(f"REM delta (REM - raw shots):                            {er - eu:.12f} Ha")
    try:
        print(
            f"\nReference Tr[H rho] (same rho, exact Pauli from DM; no shot noise): {trace_noisy_energy:.12f} Ha"
        )
    except NameError:
        pass

Shot estimate (unmitigated): -0.183076187147 Ha
Shot estimate (REM):         -0.132490343226 Ha


In [5]:
exact_simulator = cirq.Simulator(seed=random_seed)
resolved_exact = cirq.resolve_parameters(ansatz_circuit, resolver)
exact_state = exact_simulator.simulate(resolved_exact, qubit_order=ansatz_qubits).final_state_vector
exact_energy = np.vdot(exact_state, hamiltonian_matrix @ exact_state).real
print(f"Exact noiseless energy at same theta: {exact_energy:.12f} Ha")

Exact noiseless energy at same theta: -0.049584931408 Ha
